# CSH-AQI on your processed hourly data
**Integrated Air Quality and Health Risk Prediction Framework: 8 KSPCB stations, Karnataka**

This notebook trains and evaluates the **CSH-AQI** (CPCB-Structured Hybrid) forecaster directly on **your own processed hourly files** (`<station>_Final_hourly_AQI`, 1 Nov 2025 – 22 Jul 2026). These are the same files used for your earlier 1/6/12/24-hour experiments, so the results compare directly with them.

What it does:

1. **Loads your 8 hourly files** as they are: your hourly values, your 24 h / 8 h running averages, sub-indices and AQI. No extra cleaning is applied.
2. **CSH-AQI hybrid forecaster** for +1, +6, +12 and +24 h:
   * **Stage A**: for every pollutant, forecast only the *unknown* part of its CPCB averaging window (mean over hours t+1 … t+h) with gradient boosting.
   * **Stage B**: combine the already-observed hours with the forecast hours, apply the CPCB minimum-data rule and the official sub-index + max rule, giving a structural AQI forecast and the dominant pollutant.
   * **Stage C**: a residual model, trained out-of-fold, corrects what the structure misses.
   * Each component is trained on the **level** and on the **change from current conditions** ("anchored"). The final forecast is the equal-weight average of the six members, with weights fixed in advance, **not tuned on validation or test**.
3. **Evaluation**: same calendar split as your notebooks (train before 17 Apr 2026, validation 17 Apr – 29 May, test 29 May – 21 Jul), plus an h-hour gap at each boundary so no training target overlaps the next period. Validation = trained on train only. Test = refit on train + validation, scored once. Baselines are persistence ("AQI stays as now") and your previous best (Huber + Huber on a 3-hour window). Metrics are MAE, RMSE, R², AQI-category accuracy and alert F1 (AQI > 100).
4. Day-of-year (seasonal) features are switched off automatically, because these files cover less than one year.

**Requirements:** Python 3.9+, `pip install pandas numpy scikit-learn matplotlib openpyxl` (optional: `pip install lightgbm`, used automatically if installed).

**Runtime:** about 50–60 minutes on a 2-core machine, faster with LightGBM. `FAST = True` takes about 10 minutes and is for checking that everything runs.

In [ ]:
# ---- configuration -------------------------------------------------------
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")   # Windows 11: avoids the harmless 'wmic' core-count warning from joblib

DATA_DIR = "../data/processed_user"       # folder containing the 8 files <station>_Final_hourly_AQI (.csv / .xls / .xlsx)
FAST = False           # True = quick smoke test (fewer trees / folds)
OUT_DIR = "../results"
os.makedirs(OUT_DIR, exist_ok=True)

RUN_TAG = "your_hourly_files"

import os, time, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

## 1. Load your processed hourly files

In [ ]:
import glob
STATION_FILES = {                     # station -> file name without extension
    "chikkaballapura": "chikkaballapura_Final_hourly_AQI",
    "chikkamagaluru": "chikkamagaluru_Final_hourly_AQI",
    "hebbal": "hebbal_Final_hourly_AQI",
    "jayanagar": "jayanagar_Final_hourly_AQI",
    "madikeri": "madikeri_Final_hourly_AQI",
    "mysuru": "mysuru_Final_hourly_AQI",
    "ramanagara": "ramanagara_Final_hourly_AQI",
    "silkboard": "silkboard_Final_hourly_AQI",
}
AVG_COL = {"pm25": "pm25_avg_24h", "pm10": "pm10_avg_24h", "no2": "no2_avg_24h",
           "so2": "so2_avg_24h", "o3": "o3_avg_8h", "co": "co_avg_8h"}


def read_any(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.read_excel(path)


frames = []
for st, stem in STATION_FILES.items():
    hits = sorted(h for ext in ("csv", "xls", "xlsx") for h in glob.glob(os.path.join(DATA_DIR, f"*{stem}.{ext}")))
    if not hits:
        raise FileNotFoundError(f"{stem}.csv / .xls / .xlsx not found in {os.path.abspath(DATA_DIR)}")
    d = read_any(hits[0])
    out = pd.DataFrame({"datetime": pd.to_datetime(d["datetimeLocal"].astype(str).str[:19]), "station": st})
    for c in ["pm25", "pm10", "no2", "so2", "o3", "co", "no", "nox", "temperature", "relativehumidity", "AQI"]:
        out[c] = pd.to_numeric(d[c], errors="coerce")
    for p, col in AVG_COL.items():                 # your CPCB running averages and sub-indices
        out[p + "_avg"] = pd.to_numeric(d[col], errors="coerce")
        out[p + "_si"] = pd.to_numeric(d[p + "_subindex"], errors="coerce")
    frames.append(out)
    print(f"{st:16s} <- {os.path.basename(hits[0])}  ({len(out)} rows)")

hourly = pd.concat(frames, ignore_index=True).sort_values(["station", "datetime"]).reset_index(drop=True)
hourly_model = hourly

g = hourly.groupby("station")
display(pd.DataFrame({"start": g.datetime.min(), "end": g.datetime.max(), "hours": g.size(),
                      "AQI median": g.AQI.median(), "AQI max": g.AQI.max(),
                      "AQI missing %": g.AQI.apply(lambda s: s.isna().mean() * 100).round(1)}))

## 2. CPCB helpers, CSH-AQI model and baselines

In [ ]:
import numpy as np
import pandas as pd

POLLUTANTS = ["pm25", "pm10", "no2", "so2", "o3", "co"]          # used in AQI
OTHER_GAS = ["no", "nox"]
MET = ["temperature", "relativehumidity", "wind_speed", "wind_direction"]
ALL_VARS = POLLUTANTS + OTHER_GAS + MET

SENTINELS = [999.99, -99.99, 9999, -999, 999, -9999]

# CPCB breakpoints. The top band ("Severe", 401-500) is given a finite upper
# concentration so the sub-index keeps rising above 400 (capped at 500) instead
# of being stuck at exactly 400.
BREAKPOINTS = {
    "pm25": [(0, 30, 0, 50), (30, 60, 50, 100), (60, 90, 100, 200), (90, 120, 200, 300), (120, 250, 300, 400), (250, 380, 400, 500)],
    "pm10": [(0, 50, 0, 50), (50, 100, 50, 100), (100, 250, 100, 200), (250, 350, 200, 300), (350, 430, 300, 400), (430, 510, 400, 500)],
    "no2":  [(0, 40, 0, 50), (40, 80, 50, 100), (80, 180, 100, 200), (180, 280, 200, 300), (280, 400, 300, 400), (400, 520, 400, 500)],
    "so2":  [(0, 40, 0, 50), (40, 80, 50, 100), (80, 380, 100, 200), (380, 800, 200, 300), (800, 1600, 300, 400), (1600, 2400, 400, 500)],
    "o3":   [(0, 50, 0, 50), (50, 100, 50, 100), (100, 168, 100, 200), (168, 208, 200, 300), (208, 748, 300, 400), (748, 1000, 400, 500)],
    "co":   [(0, 1.0, 0, 50), (1.0, 2.0, 50, 100), (2.0, 10.0, 100, 200), (10.0, 17.0, 200, 300), (17.0, 34.0, 300, 400), (34.0, 51.0, 400, 500)],
}
CATEGORIES = [(50, "Good"), (100, "Satisfactory"), (200, "Moderate"), (300, "Poor"), (400, "Very Poor"), (np.inf, "Severe")]


def subindex(conc, pollutant):
    """Vectorised CPCB sub-index (linear interpolation inside each band)."""
    conc = np.asarray(conc, dtype=float)
    out = np.full(conc.shape, np.nan)
    for blo, bhi, ilo, ihi in BREAKPOINTS[pollutant]:
        m = (conc >= blo) & (conc <= bhi) & np.isnan(out)
        out[m] = ilo + (ihi - ilo) / (bhi - blo) * (conc[m] - blo)
    top = BREAKPOINTS[pollutant][-1]
    out[conc > top[1]] = 500.0
    return out


def aqi_category(aqi):
    aqi = np.asarray(aqi, dtype=float)
    out = np.full(aqi.shape, None, dtype=object)
    lo = -np.inf
    for hi, name in CATEGORIES:
        out[(aqi > lo) & (aqi <= hi)] = name
        lo = hi
    return out


def aqi_from_averages(avgs):
    """avgs: dict pollutant -> array of CPCB-averaged concentration.
    Returns (AQI, subindex dict). AQI = max sub-index, valid only if >=3
    pollutants available and at least one of PM2.5/PM10 (CPCB rule)."""
    si = {p: subindex(avgs[p], p) for p in POLLUTANTS}
    S = np.column_stack([si[p] for p in POLLUTANTS])
    n_avail = np.isfinite(S).sum(1)
    pm_ok = np.isfinite(si["pm25"]) | np.isfinite(si["pm10"])
    with np.errstate(all="ignore"):
        aqi = np.nanmax(np.where(np.isfinite(S), S, -np.inf), axis=1)
    aqi[(n_avail < 3) | ~pm_ok] = np.nan
    return aqi, si




"""
Step 2 - CPCB-Structured Hybrid forecaster (CSH-AQI) + baselines.

Idea
----
CPCB AQI at time t+h is not an arbitrary number: it is
    AQI(t+h) = max_p  SI_p( running-average_p(t+h) )
where the running average is 24 h for PM2.5/PM10/NO2/SO2 and 8 h for O3/CO.
For a forecast made at time t, part of every running window is ALREADY
OBSERVED (hours t+h-23 ... t) and only the last h hours are unknown.

The hybrid therefore
  Stage A  forecasts, for every pollutant, only the unknown part of its window
           (mean concentration over hours t+1 ... t+h) with a gradient-boosted
           model,
  Stage B  rebuilds the running averages exactly (known part + forecast part)
           and applies the official CPCB sub-index + max rule -> "structural"
           AQI forecast, plus the dominant pollutant,
  Stage C  a second gradient-boosted model learns the residual
           (true AQI - structural AQI) from the structural outputs and context
           features (out-of-fold, so it never sees its own training fit).
Final forecast = structural AQI + residual correction.

Baselines: persistence, a replica of the existing best (Huber on 3 h window),
and a direct gradient-boosted AQI model with the same features (ablation).
"""
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import HuberRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


try:                                   # use LightGBM when available (same algorithm family)
    import lightgbm as lgb
    HAVE_LGB = True
except Exception:
    HAVE_LGB = False

HORIZONS = [1, 6, 12, 24]
AVG24 = ["pm25", "pm10", "no2", "so2"]
AVG8 = ["o3", "co"]


def gbm(seed=0, loss="squared_error", n_iter=400, lr=0.05):
    if HAVE_LGB:
        return lgb.LGBMRegressor(n_estimators=n_iter, learning_rate=lr, num_leaves=31,
                                 min_child_samples=40, subsample=0.8, subsample_freq=1,
                                 colsample_bytree=0.8, reg_lambda=1.0, random_state=seed,
                                 objective="regression" if loss == "squared_error" else "huber",
                                 verbose=-1)
    return HistGradientBoostingRegressor(loss=loss, max_iter=n_iter, learning_rate=lr,
                                         max_leaf_nodes=31, min_samples_leaf=40,
                                         l2_regularization=1.0, max_features=0.8,
                                         early_stopping=False, random_state=seed)


# ---------------------------------------------------------------------------
# Feature engineering (all features use information available at time t only)
# ---------------------------------------------------------------------------
def _station_base(s):
    """s: one station, hourly grid, sorted. Returns base feature frame."""
    f = pd.DataFrame(index=s.index)
    conc = POLLUTANTS + ["no", "nox"]
    for c in conc:
        x = s[c]
        f[c] = x
        for L in (1, 2, 3, 6, 12, 23):
            f[f"{c}_lag{L}"] = x.shift(L)
        for w in (3, 6, 12, 24):
            f[f"{c}_mean{w}"] = x.rolling(w, min_periods=max(1, w // 2)).mean()
        f[f"{c}_std24"] = x.rolling(24, min_periods=12).std()
        f[f"{c}_d1"] = x - x.shift(1)
        f[f"{c}_d3"] = x - x.shift(3)
    for p in POLLUTANTS:
        f[f"{p}_avg"] = s[f"{p}_avg"]
        f[f"{p}_si"] = s[f"{p}_si"]
    a = s["AQI"]
    f["AQI"] = a
    for L in (1, 2, 3, 6, 12, 24, 48, 168):
        f[f"AQI_lag{L}"] = a.shift(L)
    f["AQI_d1"], f["AQI_d3"], f["AQI_d24"] = a - a.shift(1), a - a.shift(3), a - a.shift(24)
    f["AQI_std24"] = a.rolling(24, min_periods=12).std()
    for c in ["temperature", "relativehumidity", "wind_speed", "wind_u", "wind_v"]:
        x = s[c] if c in s else pd.Series(np.nan, index=s.index)
        f[c] = x
        f[f"{c}_lag1"], f[f"{c}_lag2"] = x.shift(1), x.shift(2)
        f[f"{c}_mean6"] = x.rolling(6, min_periods=3).mean()
        f[f"{c}_mean24"] = x.rolling(24, min_periods=12).mean()
    f["temp_d24"] = f["temperature"] - f["temperature"].shift(24)
    f["rh_d24"] = f["relativehumidity"] - f["relativehumidity"].shift(24)
    if "real_pm" in s:
        f["real_share24"] = s["real_pm"].rolling(24, min_periods=1).mean()
    t = s.index
    f["hour_sin"], f["hour_cos"] = np.sin(2 * np.pi * t.hour / 24), np.cos(2 * np.pi * t.hour / 24)
    f["dow"] = t.dayofweek
    f["weekend"] = (t.dayofweek >= 5).astype(int)
    f["doy_sin"], f["doy_cos"] = np.sin(2 * np.pi * t.dayofyear / 365.25), np.cos(2 * np.pi * t.dayofyear / 365.25)
    return f


def _station_horizon(s, h):
    """Horizon-specific features, structural 'known parts' and targets."""
    f = pd.DataFrame(index=s.index)
    tt = s.index + pd.Timedelta(hours=h)
    f["tgt_hour_sin"], f["tgt_hour_cos"] = np.sin(2 * np.pi * tt.hour / 24), np.cos(2 * np.pi * tt.hour / 24)
    f["tgt_dow"] = tt.dayofweek
    known, unknown_target = {}, {}
    for p in POLLUTANTS:
        x = s[p]
        W = 24 if p in AVG24 else 8
        K = max(0, W - h)                 # observed hours that remain in the window at t+h
        L = min(h, W)                     # hours of the window that are still in the future
        if K > 0:
            known[p] = x.rolling(K, min_periods=1).mean()
            f[f"{p}_known_mean"] = known[p]
            f[f"{p}_known_n"] = x.notna().astype(float).rolling(K, min_periods=1).sum()
        else:
            known[p] = pd.Series(np.nan, index=s.index)
        # same window yesterday / last week (fully observed at time t)
        f[f"{p}_win_yday"] = x.rolling(L, min_periods=max(1, L // 2)).mean().shift(24 - h)
        f[f"{p}_at_tgt_yday"] = x.shift(24 - h) if h <= 24 else np.nan
        f[f"{p}_at_tgt_wk"] = x.shift(168 - h)
        # Stage-A target: mean of the future (unknown) hours of the window
        unknown_target[p] = x.rolling(L, min_periods=int(np.ceil(0.75 * L))).mean().shift(-h)
    f["AQI_at_tgt_yday"] = s["AQI"].shift(24 - h) if h <= 24 else np.nan
    f["AQI_at_tgt_wk"] = s["AQI"].shift(168 - h)
    y = s["AQI"].shift(-h)
    return f, known, unknown_target, y


def build_dataset(hourly, h):
    """hourly: all stations, columns incl. datetime, station, pollutants, *_avg, *_si, AQI."""
    parts = []
    for st, s in hourly.groupby("station", sort=True):
        s = s.set_index("datetime").sort_index()
        s = s[~s.index.duplicated()].asfreq("1h")
        s["station"] = st
        base = _station_base(s)
        hf, known, unk, y = _station_horizon(s, h)
        d = pd.concat([base, hf], axis=1)
        for p in POLLUTANTS:
            d[f"KNOWN_{p}"] = known[p]
            d[f"UNK_{p}"] = unk[p]
            # anchor = current 24 h level of the pollutant (for scale-free targets)
            d[f"ANCH_{p}"] = s[p].rolling(24, min_periods=6).mean()
        d["y"] = y
        d["station"] = st
        parts.append(d.reset_index().rename(columns={"index": "datetime"}))
    D = pd.concat(parts, ignore_index=True)
    D["station_code"] = D["station"].astype("category").cat.codes
    return D


def feature_columns(D):
    drop = {"datetime", "station", "y"} | {c for c in D if c.startswith(("KNOWN_", "UNK_", "ANCH_"))}
    # drop columns with no data at all (e.g. wind when a dataset has no wind sensor)
    return [c for c in D.columns if c not in drop and D[c].notna().any()]


def drop_season_if_short(cols, D_fit, min_days=330):
    """Day-of-year features are only meaningful when the training data covers a
    full annual cycle; with less, trees extrapolate the season edge to unseen
    dates. Drop them in that case."""
    span = (D_fit["datetime"].max() - D_fit["datetime"].min()).days
    return cols if span >= min_days else [c for c in cols if not c.startswith("doy")]


# ---------------------------------------------------------------------------
# Splits (same calendar boundaries as the original notebook, plus an embargo)
# ---------------------------------------------------------------------------
VAL_START = pd.Timestamp("2026-04-17 07:00")
TEST_START = pd.Timestamp("2026-05-29 14:00")
TEST_END = pd.Timestamp("2026-07-21 23:00")


def split_masks(D, h, train_from=None):
    t = D["datetime"]
    emb = pd.Timedelta(hours=h)            # target of a training row must end before validation starts
    tr = t < VAL_START - emb
    if train_from is not None:
        tr &= t >= pd.Timestamp(train_from)
    va = (t >= VAL_START) & (t < TEST_START - emb)
    te = (t >= TEST_START) & (t <= TEST_END)
    return tr.to_numpy(), va.to_numpy(), te.to_numpy()


# ---------------------------------------------------------------------------
# Structural reconstruction of CPCB AQI from forecast concentrations
# ---------------------------------------------------------------------------
def structural_aqi(D, unk_pred, h):
    avgs = {}
    for p in POLLUTANTS:
        W = 24 if p in AVG24 else 8
        K, L = max(0, W - h), min(h, W)
        kn = D[f"KNOWN_{p}"].to_numpy(float)
        fu = np.asarray(unk_pred[p], float)
        if K > 0:
            avg = np.where(np.isfinite(kn), (K * kn + L * fu) / W, fu)
            # CPCB minimum data rule (16 of 24 h, 6 of 8 h): observed hours + forecast hours
            n_obs = D[f"{p}_known_n"].to_numpy(float) if f"{p}_known_n" in D else np.full(len(D), K)
            avg = np.where(np.nan_to_num(n_obs) + L >= int(np.ceil(W * 2 / 3)), avg, np.nan)
        else:
            avg = fu
        avgs[p] = avg
    aqi, si = aqi_from_averages(avgs)
    # fall back to max of available sub-indices if CPCB 3-pollutant rule fails in forecast space
    S = np.column_stack([si[p] for p in POLLUTANTS])
    with np.errstate(all="ignore"):
        any_aqi = np.nanmax(np.where(np.isfinite(S), S, -np.inf), axis=1)
    aqi = np.where(np.isfinite(aqi), aqi, np.where(np.isfinite(any_aqi), any_aqi, np.nan))
    return aqi, si, avgs


class CPCBStructuredHybrid:
    """Stage A (per-pollutant future-window forecasters) -> Stage B (exact CPCB
    reconstruction) -> Stage C (residual corrector trained out-of-fold)."""

    def __init__(self, h, n_folds=4, n_iter=300, seed=0, anchored=True):
        self.h, self.n_folds, self.n_iter, self.seed = h, n_folds, n_iter, seed
        self.anchored = anchored
        self.fill = {}

    def _anchor(self, D, p):
        a = D[f"ANCH_{p}"].to_numpy(float)
        return np.where(np.isfinite(a), a, self.fill.get(p, np.nan))

    def _fit_stage_a(self, X, D):
        models = {}
        for p in POLLUTANTS:
            yt = D[f"UNK_{p}"].to_numpy(float)
            m = np.isfinite(yt)
            if m.sum() < 500:
                models[p] = None
                continue
            target = np.log1p(np.clip(yt, 0, None))
            if self.anchored:            # forecast log-ratio of future level to current level
                target = target - np.log1p(self._anchor(D, p))
                m &= np.isfinite(target)
            mdl = gbm(self.seed, n_iter=self.n_iter)
            mdl.fit(X[m], target[m])
            models[p] = mdl
        return models

    def _predict_stage_a(self, models, X, D):
        out = {}
        for p in POLLUTANTS:
            if models[p] is None:
                out[p] = D[f"KNOWN_{p}"].to_numpy(float)  # fallback: assume level persists
            else:
                z = models[p].predict(X)
                if self.anchored:
                    z = z + np.log1p(self._anchor(D, p))
                out[p] = np.expm1(z)
        return out

    def _stage_c_features(self, X, D, unk_pred):
        aqi_s, si, avgs = structural_aqi(D, unk_pred, self.h)
        extra = [aqi_s] + [si[p] for p in POLLUTANTS] + [avgs[p] for p in POLLUTANTS] + [unk_pred[p] for p in POLLUTANTS]
        return np.column_stack([X] + extra), aqi_s

    def fit(self, D_tr, cols):
        X = D_tr[cols].to_numpy(np.float32)
        self.fill = {p: float(np.nanmedian(D_tr[f"ANCH_{p}"])) for p in POLLUTANTS}
        # out-of-fold Stage-A predictions on the training set (contiguous time blocks)
        order = np.argsort(D_tr["datetime"].to_numpy())
        folds = np.array_split(order, self.n_folds)
        oof = {p: np.full(len(D_tr), np.nan) for p in POLLUTANTS}
        for k in range(self.n_folds):
            hold = folds[k]
            fit_idx = np.concatenate([folds[j] for j in range(self.n_folds) if j != k])
            mods = self._fit_stage_a(X[fit_idx], D_tr.iloc[fit_idx])
            pr = self._predict_stage_a(mods, X[hold], D_tr.iloc[hold])
            for p in POLLUTANTS:
                oof[p][hold] = pr[p]
        self.stage_a = self._fit_stage_a(X, D_tr)
        Xc, aqi_s = self._stage_c_features(X, D_tr, oof)
        y = D_tr["y"].to_numpy(float)
        m = np.isfinite(y) & np.isfinite(aqi_s)
        self.stage_c = gbm(self.seed + 1, n_iter=self.n_iter)
        self.stage_c.fit(Xc[m], (y - aqi_s)[m])
        return self

    def predict(self, D, cols, return_parts=False):
        X = D[cols].to_numpy(np.float32)
        unk = self._predict_stage_a(self.stage_a, X, D)
        Xc, aqi_s = self._stage_c_features(X, D, unk)
        final = aqi_s + self.stage_c.predict(Xc)
        final = np.clip(final, 0, 500)
        if return_parts:
            _, si, _ = structural_aqi(D, unk, self.h)
            S = np.column_stack([si[p] for p in POLLUTANTS])
            dom = np.array(POLLUTANTS, dtype=object)[np.argmax(np.where(np.isfinite(S), S, -1), axis=1)]
            return final, aqi_s, dom
        return final


# ---------------------------------------------------------------------------
# Baselines
# ---------------------------------------------------------------------------
HUBER_FEATS = ["pm25", "pm10", "no", "no2", "nox", "co", "o3", "so2", "temperature", "relativehumidity", "AQI"]


def huber3h_features(D):
    """Same inputs as the existing notebook: 11 variables at t-2, t-1, t."""
    cols = []
    for c in HUBER_FEATS:
        cols += [f"{c}_lag2", f"{c}_lag1", c]
    return D[cols].to_numpy(float)


class Huber3h:
    """Replica of the existing best model: two Huber regressors on the last 3 hours, averaged."""

    def fit(self, D_tr):
        X = huber3h_features(D_tr)
        y = D_tr["y"].to_numpy(float)
        self.imp = SimpleImputer(strategy="median").fit(X)
        self.sc = StandardScaler().fit(self.imp.transform(X))
        Xs = self.sc.transform(self.imp.transform(X))
        m = np.isfinite(y)
        self.m1 = HuberRegressor(epsilon=1.35, alpha=1e-4, max_iter=500).fit(Xs[m], y[m])
        self.m2 = HuberRegressor(epsilon=1.5, alpha=1e-3, max_iter=500).fit(Xs[m], y[m])
        return self

    def predict(self, D):
        Xs = self.sc.transform(self.imp.transform(huber3h_features(D)))
        return (self.m1.predict(Xs) + self.m2.predict(Xs)) / 2


class DirectGBM:
    """Gradient boosting on AQI directly. anchored=True learns the change
    AQI(t+h) - AQI(t) instead of the level."""

    def __init__(self, n_iter=300, seed=0, anchored=False):
        self.n_iter, self.seed, self.anchored = n_iter, seed, anchored

    def fit(self, D_tr, cols):
        y = D_tr["y"].to_numpy(float)
        if self.anchored:
            y = y - D_tr["AQI"].to_numpy(float)
        m = np.isfinite(y)
        self.m = gbm(self.seed, n_iter=self.n_iter).fit(D_tr[cols].to_numpy(np.float32)[m], y[m])
        return self

    def predict(self, D, cols):
        p = self.m.predict(D[cols].to_numpy(np.float32))
        if self.anchored:
            p = p + D["AQI"].to_numpy(float)
        return np.clip(p, 0, 500)


# ---------------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------------
CAT_ORDER = ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"]


def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    cy, cp = aqi_category(y), aqi_category(p)
    alert_y, alert_p = y > 100, p > 100            # Moderate or worse: risk for sensitive groups
    tp = (alert_y & alert_p).sum()
    prec = tp / max(alert_p.sum(), 1)
    rec = tp / max(alert_y.sum(), 1)
    return {
        "MAE": mean_absolute_error(y, p),
        "RMSE": float(np.sqrt(mean_squared_error(y, p))),
        "R2": r2_score(y, p),
        "Cat_Acc": float((cy == cp).mean()),
        "Alert_F1": float(2 * prec * rec / max(prec + rec, 1e-9)),
    }

## 3. Experiment: validation (model selection), then refit and score test once

In [ ]:
MEMBERS = ["Structural", "Structural (anchored)", "Hybrid", "Hybrid (anchored)", "Direct GBM", "Direct GBM (anchored)"]
N_ITER, N_FOLDS = (60, 2) if FAST else (300, 4)
_gbm_orig = gbm
def gbm(seed=0, loss="squared_error", n_iter=400, lr=0.05):
    return _gbm_orig(seed, loss, n_iter=min(n_iter, N_ITER), lr=0.15 if FAST else lr)


def fit_predict(Dfit, Deval, cols, h):
    out = {"Persistence": Deval["AQI"].to_numpy(float)}
    out["Huber+Huber 3h (existing best)"] = Huber3h().fit(Dfit).predict(Deval)
    for anch in (False, True):
        tag = " (anchored)" if anch else ""
        hy = CPCBStructuredHybrid(h, anchored=anch, n_folds=N_FOLDS).fit(Dfit, cols)
        f, s, dom = hy.predict(Deval, cols, return_parts=True)
        out["Structural" + tag] = np.clip(s, 0, 500)
        out["Hybrid" + tag] = f
        out["Direct GBM" + tag] = DirectGBM(anchored=anch).fit(Dfit, cols).predict(Deval, cols)
        if anch:
            out["pred_dominant"] = dom
    out["CSH-AQI ensemble"] = np.mean([out[m] for m in MEMBERS], axis=0)
    return out


print(f"Run: {RUN_TAG}  |  modelling data {hourly_model.datetime.min()} to {hourly_model.datetime.max()}  ({len(hourly_model)} station-hours)")

results, preds = [], {}
for h in HORIZONS:
    t0 = time.time()
    D = build_dataset(hourly_model, h)
    cols = feature_columns(D)
    tr, va, te = split_masks(D, h)
    yok = np.isfinite(D["y"]).to_numpy()
    ev = yok & np.isfinite(D["AQI"]).to_numpy()
    trval = (D["datetime"] < TEST_START - pd.Timedelta(hours=h)).to_numpy()
    preds[h] = {}
    for phase, fit_mask, ev_mask in [("val", tr & yok, va & ev), ("test", trval & yok, te & ev)]:
        Dfit, Dev = D[fit_mask], D[ev_mask]
        o = fit_predict(Dfit, Dev, drop_season_if_short(cols, Dfit), h)
        P = pd.DataFrame({"datetime": Dev["datetime"].values, "station": Dev["station"].values, "y": Dev["y"].values, **o})
        preds[h][phase] = P
        for m in [k for k in o if k != "pred_dominant"]:
            results.append({"horizon": h, "split": phase, "model": m, "n": len(P), **metrics(P["y"], P[m])})
    print(f"+{h} h done in {time.time()-t0:.0f} s")

R = pd.DataFrame(results)
R.to_csv(os.path.join(OUT_DIR, f"csh_aqi_results_{RUN_TAG}.csv"), index=False)
with open(os.path.join(OUT_DIR, f"csh_aqi_predictions_{RUN_TAG}.pkl"), "wb") as fh:
    pickle.dump(preds, fh)

## 4. Results

In [ ]:
show = ["Persistence", "Huber+Huber 3h (existing best)", "Direct GBM", "Structural", "Hybrid", "CSH-AQI ensemble"]
for split in ["val", "test"]:
    t = R[(R.split == split) & R.model.isin(show)].copy()
    base = t[t.model == "Persistence"].set_index("horizon")["MAE"]
    t["MAE skill vs persistence %"] = (100 * (1 - t.MAE / t.horizon.map(base))).round(1)
    t = t.set_index(["horizon", "model"])[["MAE", "RMSE", "R2", "Cat_Acc", "Alert_F1", "MAE skill vs persistence %"]].round(3)
    print(f"\n==== {split.upper()} ====")
    display(t)

In [ ]:
# One table per horizon (same layout as the earlier result images) -> Excel, one sheet per horizon
ORDER = [("Persistence", "Baseline"), ("Huber+Huber 3h (existing best)", "Existing best"),
         ("Direct GBM", "Component"), ("Direct GBM (anchored)", "Component"),
         ("Structural", "Component"), ("Structural (anchored)", "Component"),
         ("Hybrid", "Component"), ("Hybrid (anchored)", "Component"),
         ("CSH-AQI ensemble", "PROPOSED (final)")]
with pd.ExcelWriter(os.path.join(OUT_DIR, f"CSH_AQI_metrics_by_horizon_{RUN_TAG}.xlsx")) as w:
    for h in HORIZONS:
        rows = []
        for m, cat in ORDER:
            v = R[(R.horizon == h) & (R.model == m) & (R.split == "val")].iloc[0]
            t = R[(R.horizon == h) & (R.model == m) & (R.split == "test")].iloc[0]
            rows.append({"Model": m, "Category": cat,
                         "Val MAE": v.MAE, "Val RMSE": v.RMSE, "Val R2": v.R2,
                         "Test MAE": t.MAE, "Test RMSE": t.RMSE, "Test R2": t.R2,
                         "Test category accuracy": t.Cat_Acc})
        table = pd.DataFrame(rows).round(4)
        table.to_excel(w, sheet_name=f"{h}-hour", index=False)
        print(f"\n===== {h}-HOUR AQI FORECASTING =====")
        display(table)

In [ ]:
# Per-station MAE on the test period
rows = []
for h in HORIZONS:
    P = preds[h]["test"]
    for st, s in P.groupby("station"):
        rows.append({"horizon": h, "station": st,
                     "Persistence": (s["Persistence"] - s.y).abs().mean(),
                     "Existing best": (s["Huber+Huber 3h (existing best)"] - s.y).abs().mean(),
                     "CSH-AQI": (s["CSH-AQI ensemble"] - s.y).abs().mean()})
station_table = pd.DataFrame(rows).set_index(["horizon", "station"]).round(2)
display(station_table.unstack(0))

## 5. Figures

In [ ]:
C_NEW, C_OLD, C_BASE, INK, MUTED = "#2a78d6", "#eb6834", "#8a8984", "#0b0b0b", "#52514e"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#c9c8c3",
                     "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED, "font.size": 10})

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
series = [("Persistence", "Persistence", C_BASE, "--"),
          ("Huber+Huber 3h (existing best)", "Existing best (Huber+Huber)", C_OLD, "-"),
          ("CSH-AQI ensemble", "CSH-AQI (proposed)", C_NEW, "-")]
for ax, split in zip(axes, ["val", "test"]):
    for m, label, col, ls in series:
        d = R[(R.split == split) & (R.model == m)].sort_values("horizon")
        ax.plot(d.horizon, d.MAE, ls, color=col, lw=2, marker="o", ms=8, label=label)
        if m == "CSH-AQI ensemble":
            ax.annotate(f"{d.MAE.iloc[-1]:.1f}", (d.horizon.iloc[-1], d.MAE.iloc[-1]), xytext=(9, 0),
                        textcoords="offset points", ha="left", va="center", color=C_NEW, fontsize=9, fontweight="bold")
    ax.set_xticks(HORIZONS); ax.set_xticklabels([f"+{h} h" for h in HORIZONS]); ax.set_xlim(-0.5, 27.5)
    ax.set_title({"val": "Validation (17 Apr – 29 May 2026)", "test": "Test (29 May – 21 Jul 2026)"}[split], color=INK, loc="left")
    ax.set_ylabel("MAE (AQI points)"); ax.grid(axis="y", color="#ecebe7"); ax.set_ylim(bottom=0)
h_, l_ = axes[0].get_legend_handles_labels()
fig.legend(h_, l_, frameon=False, loc="lower center", ncol=3)
fig.suptitle("Forecast error by horizon - lower is better", x=0.01, ha="left", color=INK, fontweight="bold")
fig.tight_layout(rect=(0, 0.08, 1, 1)); fig.savefig(os.path.join(OUT_DIR, f"fig_mae_by_horizon_{RUN_TAG}.png"), dpi=200); plt.show()

In [ ]:
# Example: +12 h forecasts at Silk Board during the test period
P = preds[12]["test"]; s = P[P.station == "silkboard"].sort_values("datetime")
s = s.set_index("datetime")
tt = s.index + pd.Timedelta(hours=12)          # plot at the time being forecast
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(tt, s.y, color=INK, lw=1.6, label="Observed AQI")
ax.plot(tt, s["Huber+Huber 3h (existing best)"], color=C_OLD, lw=1.2, alpha=.9, label="Existing best")
ax.plot(tt, s["CSH-AQI ensemble"], color=C_NEW, lw=1.6, label="CSH-AQI (proposed)")
for lvl in (50, 100, 200):
    ax.axhline(lvl, color="#d9d8d3", lw=1, zorder=0)
ax.set_ylabel("AQI"); ax.legend(frameon=False, ncol=3, loc="upper right")
ax.set_title("Silk Board - AQI forecast 12 hours ahead (test period)", loc="left", color=INK, fontweight="bold")
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, f"fig_silkboard_12h_{RUN_TAG}.png"), dpi=200); plt.show()

## 6. Health-risk layer
The forecast AQI maps to the CPCB category and its health advisory. Because Stage B rebuilds every sub-index, the model also forecasts the **dominant pollutant**, which tells you *why* the air will be bad, not just *how* bad.

In [ ]:
ADVISORY = {
    "Good": "Minimal impact.",
    "Satisfactory": "Minor breathing discomfort to sensitive people.",
    "Moderate": "Breathing discomfort to people with lung disease (e.g. asthma), and discomfort to people with heart disease, children and older adults.",
    "Poor": "Breathing discomfort to most people on prolonged exposure.",
    "Very Poor": "Respiratory illness on prolonged exposure.",
    "Severe": "Affects healthy people and seriously impacts those with existing diseases.",
}
P = preds[24]["test"]
last = P.sort_values("datetime").groupby("station").tail(1).copy()
last["forecast_for"] = last["datetime"] + pd.Timedelta(hours=24)
last["forecast_AQI"] = last["CSH-AQI ensemble"].round()
last["category"] = aqi_category(last["forecast_AQI"])
last["dominant"] = last["pred_dominant"]
last["advisory"] = last["category"].map(ADVISORY)
display(last[["station", "forecast_for", "forecast_AQI", "category", "dominant", "advisory", "y"]]
        .rename(columns={"y": "observed_AQI"}).reset_index(drop=True))